# 표본이 숨긴 것을 전량이 드러낸다 — 공공데이터포털 대량 수집

> `notebooks/01-데이터수집/06.표본이숨긴것을전량이드러낸다.ipynb` · 2026-09-03 · 이동원
> 앞 노트북 [`05.종목의신원을잇다`](05.종목의신원을잇다.ipynb) · 설계
> [`docs/데이터파트/version3.2/공공데이터포털_수집_설계.md`](../../docs/데이터파트/version3.2/공공데이터포털_수집_설계.md)

---

## 이 노트북이 답하는 것

> **"표본에서 전부 통과한 검사가, 전량에서도 통과할까?"**

앞 노트북에서 종목 신원을 **3일치 7,000행 · 법인 5곳 82행**만 받아 검증했고
**전부 통과**했습니다. 이번에 전량을 받았습니다.

| | 표본 | 전량 |
|---|---|---|
| `stock_identity` | 7,000행 · 3일 | **4,197,242행 · 1,635일** |
| `corp_profile` | 82행 · 법인 5곳 | **40,569행 · 법인 3,141곳** |
| 검증 판정 | ✅ 이상 없음 | 🔴 **세 곳이 뒤집혔다** |

같은 검증기, 같은 코드입니다. **자료의 양만 달랐습니다.**

이 노트북은 그 셋이 무엇이었고 왜 표본에서는 안 보였는지를 직접 세어 보입니다.

---

## 0. 준비 — 저장소를 읽기 전용으로 연다

수집은 이미 끝났습니다. 여기서는 **재지만 하고 고치지 않습니다.**

In [1]:
import collections
import sqlite3
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] if Path.cwd().name.startswith("0") else Path.cwd()))

DB = Path("data/krx_cache.db")
if not DB.exists():                       # 노트북을 어디서 열든 같은 파일을 보게 한다
    DB = Path.cwd().parents[1] / "data" / "krx_cache.db"

conn = sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True)


def 하나(sql, *args):
    return conn.execute(sql, args).fetchone()


def 표(sql, *args):
    return conn.execute(sql, args).fetchall()


신원 = 하나("SELECT COUNT(*), COUNT(DISTINCT bas_dd), COUNT(DISTINCT code) FROM stock_identity")
개요 = 하나("SELECT COUNT(*), COUNT(DISTINCT crno) FROM corp_profile")
print(f"stock_identity : {신원[0]:>10,}행 · {신원[1]:,}일 · {신원[2]:,}종목")
print(f"corp_profile   : {개요[0]:>10,}행 · 법인 {개요[1]:,}곳")
print(f"DB             : {DB.stat().st_size / 1024 / 1024:,.0f} MB")

stock_identity :  4,197,242행 · 1,635일 · 3,165종목
corp_profile   :     40,569행 · 법인 3,141곳
DB             : 2,979 MB


---

## 1. 첫 번째 뒤집힘 — "포털은 외국기업을 안 준다" 는 틀렸다

표본(8일)에서 시세와 못 이은 종목을 갈랐더니 **우선주 120 · 외국기업 21 · 그 밖 0** 이었습니다.
거기서 *"포털은 우선주와 외국기업을 안 준다"* 고 적었습니다.

전량으로 다시 세어 봅니다.

In [2]:
시세 = {r[0] for r in 표("SELECT DISTINCT code FROM daily_price WHERE bas_dd >= '20200102'")}
신원코드 = {r[0] for r in 표("SELECT DISTINCT code FROM stock_identity")}

못이은 = 시세 - 신원코드
외국 = {c for c in 못이은 if c.startswith(("900", "950"))}

시세외국 = {r[0] for r in 표(
    "SELECT DISTINCT code FROM daily_price "
    "WHERE bas_dd >= '20200102' AND (code LIKE '900%' OR code LIKE '950%')")}
신원외국 = {c for c in 신원코드 if c.startswith(("900", "950"))}
첫날외국 = {r[0] for r in 표(
    "SELECT DISTINCT code FROM stock_identity "
    "WHERE bas_dd = '20200102' AND (code LIKE '900%' OR code LIKE '950%')")}

print(f"시세에 있는 외국계          : {len(시세외국)}종")
print(f"포털이 전 구간에서 준 외국계 : {len(신원외국)}종   ← 안 주는 게 아니다")
print(f"  그중 20200102 목록에는    : {len(첫날외국)}종   ← 표본은 여기만 봤다")
print(f"끝내 못 이은 외국계         : {len(외국)}종  {sorted(외국)}")

시세에 있는 외국계          : 27종
포털이 전 구간에서 준 외국계 : 21종   ← 안 주는 게 아니다
  그중 20200102 목록에는    : 0종   ← 표본은 여기만 봤다
끝내 못 이은 외국계         : 6종  ['900040', '900080', '900140', '900280', '950110', '950180']


🔴 **표본이 하필 그 목록을 봤을 뿐이었습니다.**

`basDt=20200102` 목록에는 외국기업이 **0종**이라, 그날만 보면 "포털이 안 준다" 로 읽힙니다.
전 구간을 보면 21종을 줍니다. 끝내 안 나오는 6종만 진짜 결측입니다.

그 6종이 언제까지 거래됐는지 봅니다 — **아직 상장 중인데 없는 것**이 있으면 그건 결함입니다.

In [3]:
print("끝내 못 이은 외국계 6종")
for 코드 in sorted(외국):
    이름, 처음, 마지막, 일수 = 하나(
        "SELECT MAX(name), MIN(bas_dd), MAX(bas_dd), COUNT(*) FROM daily_price WHERE code = ?",
        코드)
    끝 = "🔴 아직 거래 중" if 마지막 >= "20260825" else "상장폐지"
    print(f"  {코드}  {이름:14s} {처음} ~ {마지막}  {일수:>5,}일  {끝}")

끝내 못 이은 외국계 6종
  900040  차이나그레이트        20100104 ~ 20200521  2,561일  상장폐지
  900080  중국엔진집단         20100104 ~ 20210611  2,824일  상장폐지
  900140  코라오홀딩스         20101130 ~ 20260901  3,874일  🔴 아직 거래 중
  900280  골든센츄리          20161019 ~ 20250107  2,020일  상장폐지
  950110  SBI핀테크솔루션즈     20121217 ~ 20250723  3,093일  상장폐지
  950180  SNK            20190507 ~ 20220517    750일  상장폐지


---

## 2. 두 번째 뒤집힘 — "해당 없음" 이 서기 1년으로 들어왔다

표본(법인 5곳)에서 *"KRX 개장(1956-03)보다 이른 상장일 0"* 이었습니다.
전량(법인 3,142곳)에서 **51행**이 걸렸습니다.

처음에는 두 자리 연도를 1900대로 잘못 푼 줄 알았습니다. **아니었습니다.**

In [4]:
# 지금은 고쳐진 뒤라 DB 에 남아 있지 않다. 정규화 함수에 원문을 그대로 넣어 재현한다.
from ingest.clients.data_go_kr import normalize_date

원문들 = [
    ("00010101", "포털의 '해당 없음' 자리표시자 — 유가 51행 · 코스닥 42행"),
    ("11111111", "쓰레기값 1행 ((주)케이씨씨 본부영업소)"),
    ("18970925", "🔴 진짜다 — 동화약품, 국내 최고령 등록법인"),
    ("19560303", "KRX 개장일"),
    ("76/03/24", "두 자리 연도 (1976)"),
]
print(f"{'원문':<12} {'정규화 결과':<12} 무엇인가")
print("-" * 78)
for 원문, 설명 in 원문들:
    print(f"{원문:<12} {str(normalize_date(원문)):<12} {설명}")

원문           정규화 결과       무엇인가
------------------------------------------------------------------------------
00010101     None         포털의 '해당 없음' 자리표시자 — 유가 51행 · 코스닥 42행
11111111     None         쓰레기값 1행 ((주)케이씨씨 본부영업소)
18970925     18970925     🔴 진짜다 — 동화약품, 국내 최고령 등록법인
19560303     19560303     KRX 개장일
76/03/24     19760324     두 자리 연도 (1976)


**바닥선을 어디에 둘 것인가** 가 이 고침의 전부였습니다.

- `1900` 으로 올리면 → 자리표시자는 사라지지만 **동화약품(1897)이 함께 죽습니다**
- `0001` 만 콕 집으면 → 동화약품은 살지만 **`11111111` 이 남습니다**
- `1800` → 둘을 정확히 가릅니다

**거르는 것과 잃는 것을 가르는 자리**라 상수(`_EARLIEST_PLAUSIBLE`)로 못 박고
테스트로 지킵니다.

고친 것이 옳았는지는 **다른 경로**로 확인합니다 — 자리표시자를 걸러 낸 뒤의
실제 최솟값이 KRX 개장일과 맞는지 봅니다.

In [5]:
최초유가 = 하나("SELECT MIN(xchg_lstg_dt) FROM corp_profile WHERE xchg_lstg_dt IS NOT NULL")[0]
최초코스닥 = 하나("SELECT MIN(kosdaq_lstg_dt) FROM corp_profile "
                "WHERE kosdaq_lstg_dt IS NOT NULL")[0]
남은자리표시자 = 하나(
    "SELECT COUNT(*) FROM corp_profile "
    "WHERE xchg_lstg_dt = '00010101' OR kosdaq_lstg_dt = '00010101'")[0]

print(f"가장 이른 유가증권 상장일 : {최초유가}   (KRX 개장 19560303)")
print(f"가장 이른 코스닥 상장일   : {최초코스닥}   (코스닥 개장 19960701)")
print(f"남은 자리표시자           : {남은자리표시자}행")
print()
print("🔴 기대값을 우리가 넣은 값에서 뽑으면 항등식이라 아무것도 못 잡는다.")
print("   KRX 개장일은 **도메인 사실**이라 우리 자료 밖에서 온 기대값이다.")

가장 이른 유가증권 상장일 : 19560303   (KRX 개장 19560303)
가장 이른 코스닥 상장일   : 19890105   (코스닥 개장 19960701)
남은 자리표시자           : 0행

🔴 기대값을 우리가 넣은 값에서 뽑으면 항등식이라 아무것도 못 잡는다.
   KRX 개장일은 **도메인 사실**이라 우리 자료 밖에서 온 기대값이다.


### 곁가지 — 코스닥 개장보다 이른 코스닥 상장일 97곳은 결함인가

위 출력에 **`19890105`** 가 찍혔습니다. 코스닥 개장은 **1996-07-01** 입니다.
같은 부류의 자리표시자인지 세어 봅니다.

**세는 법이 중요합니다.** 자리표시자는 한 값에 뭉치고, 진짜 자료는 **분포가 매끄럽습니다.**

In [6]:
이른것 = 표(
    "SELECT substr(kosdaq_lstg_dt, 1, 4) y, COUNT(DISTINCT crno) "
    "FROM corp_profile WHERE kosdaq_lstg_dt < '19960701' GROUP BY y ORDER BY y")
전체 = 하나("SELECT COUNT(DISTINCT crno) FROM corp_profile "
         "WHERE kosdaq_lstg_dt IS NOT NULL")[0]
합 = sum(k for _, k in 이른것)
print(f"코스닥 개장(19960701)보다 이른 상장일 : 법인 {합}곳 / {전체}곳 ({합 / 전체:.1%})")
print()
for 연, 수 in 이른것:
    print(f"  {연}  {'█' * 수}  {수}곳")

코스닥 개장(19960701)보다 이른 상장일 : 법인 97곳 / 2071곳 (4.7%)

  1989  █████  5곳
  1990  ██  2곳
  1991  ████  4곳
  1992  ██████████  10곳
  1993  ██████████████████  18곳
  1994  ███████████████████████████████████  35곳
  1995  ██████████████  14곳
  1996  █████████  9곳


**뭉치지 않고 퍼져 있습니다** — 1989년 5곳에서 1994년 35곳까지 늘다가 다시 줄어듭니다.
자리표시자라면 한 값에 97곳이 몰렸을 것입니다.

코스닥의 전신인 **협회중개시장(장외시장, 1987년 개설)** 등록일이 그대로 넘어온 것으로
보입니다. 즉 **결함이 아닙니다.**

🔴 다만 *"코스닥 상장일은 1996-07-01 이후"* 라고 가정하는 코드는 이 97곳에서 틀립니다.
검증기에 이 검사를 **넣지 않은 이유**가 여기 있습니다 — 넣었으면 97건이 매번 빨간불로
켜지고, 사람이 곧 검증기를 무시하게 됩니다.

---

## 3. 세 번째 뒤집힘 — 외국기업 20종이 한 법인번호를 공유한다

이건 검증기가 **못 잡았습니다.** 사람이 `crno` 를 훑다 눈에 걸렸습니다.

포털은 외국기업의 법인등록번호 자리에 `0000000000000` 을 줍니다.
그대로 두고 `corp_profile` 과 조인하면 어떻게 되는지 재현해 봅니다.

In [7]:
from ingest.clients.data_go_kr import normalize_crno

# 포털이 실제로 준 응답 한 줄 (딥커머스 · 홍콩 법인)
원본 = {"basDt": "20260901", "srtnCd": "A900110", "isinCd": "HK0000057197",
       "mrktCtg": "KOSDAQ", "itmsNm": "딥커머스", "crno": "0000000000000",
       "corpNm": "딥커머스리미티드"}
print("포털이 준 crno :", 원본["crno"])
print("정규화 결과    :", normalize_crno(원본["crno"]))
print()
print("정상 번호는 그대로 :", normalize_crno("1101111867948"))

포털이 준 crno : 0000000000000
정규화 결과    : None

정상 번호는 그대로 : 1101111867948


왜 이게 비싼 실수인지가 핵심입니다.

> 조인이 **0행**이 되는 실수는 결과가 비어서 눈에 띕니다.
> **틀린 짝**이 붙는 실수는 결과가 그럴듯해서 안 보입니다.

그 가짜 번호로 실제로 받아 둔 법인 개요가 있었고, **거기에 서로 다른 회사가 섞여**
있었습니다. 20개 외국기업이 전부 그 하나에 붙을 뻔했습니다.

지금은 검증기가 매번 셉니다 — **한 법인번호에 몇 종목이 달렸나.**

In [8]:
뭉침 = 표("SELECT crno, COUNT(DISTINCT code) n FROM stock_identity "
        "WHERE crno IS NOT NULL GROUP BY crno ORDER BY n DESC LIMIT 5")
print("한 법인번호를 여러 종목이 공유하는 경우 (상위 5)")
for 번호, 종수 in 뭉침:
    이름 = 하나("SELECT corp_nm FROM stock_identity WHERE crno = ? LIMIT 1", 번호)[0]
    코드들 = [r[0] for r in 표("SELECT DISTINCT code FROM stock_identity WHERE crno = ?", 번호)]
    print(f"  {번호}  {종수}종  {이름[:18]:20s} {코드들}")
print()
없는것 = 하나("SELECT COUNT(DISTINCT code) FROM stock_identity WHERE crno IS NULL")[0]
print(f"crno 가 없는 종목 : {없는것}종 (전부 외국기업 — 포털이 번호를 안 준다)")
print("🔴 정상 최대는 2종(보통주+우선주)이다. 수십 종이면 자리표시자를 의심한다.")

한 법인번호를 여러 종목이 공유하는 경우 (상위 5)
  1101111867948  2종  키움증권(주)              ['039490', '311270']
  1101110000086  1종  롯데쇼핑(주)              ['023530']
  1101110002694  1종  지에스건설(주)             ['006360']
  1101110002818  1종  한국제지(주)              ['002300']
  1101110002959  1종  (주)한화                ['000880']



crno 가 없는 종목 : 20종 (전부 외국기업 — 포털이 번호를 안 준다)
🔴 정상 최대는 2종(보통주+우선주)이다. 수십 종이면 자리표시자를 의심한다.


---

## 4. 그래서 지금 다리는 어디까지 이어지나

세 곳을 고친 뒤의 대차대조입니다. **숫자가 딱 맞아떨어지는지**가 확인입니다.

In [9]:
공통 = 시세 & 신원코드
신원만 = 신원코드 - 시세
시세만 = 시세 - 신원코드

# 신원에만 있는 것의 정체 — 시장으로 갈라 본다
시장 = collections.Counter(
    하나("SELECT market FROM stock_identity WHERE code = ? LIMIT 1", c)[0] for c in 신원만)

print(f"  공통      {len(공통):>5,}종   다리가 이어진 것")
print(f"  신원에만  {len(신원만):>5,}종   {dict(시장)}")
print(f"  시세에만  {len(시세만):>5,}종   우선주 + 외국기업")
print(f"  {'':10} ------")
print(f"  시세 {len(시세):,} = 공통 {len(공통):,} + 시세에만 {len(시세만):,}"
      f"  → {len(공통) + len(시세만) == len(시세)}")
print(f"  신원 {len(신원코드):,} = 공통 {len(공통):,} + 신원에만 {len(신원만):,}"
      f"  → {len(공통) + len(신원만) == len(신원코드)}")

  공통      2,992종   다리가 이어진 것
  신원에만    173종   {'KONEX': 173}
  시세에만    135종   우선주 + 외국기업
             ------
  시세 3,127 = 공통 2,992 + 시세에만 135  → True
  신원 3,165 = 공통 2,992 + 신원에만 173  → True


**신원에만 있는 것은 전부 KONEX** 입니다. 결함이 아니라 **범위 차이**입니다 —
우리는 KOSPI·KOSDAQ 시세만 받고 KONEX 는 안 받습니다. 포털은 KONEX 도 줍니다.

이걸 "못 이은 종목" 으로 세면 없는 버그를 쫓게 됩니다.

---

## 5. 마지막 하루는 받을 수 없다 — 이건 결함이 아니라 경계 조건이다

1,636일 중 **1,635일**이 들어오고 마지막 날에서 멈췄습니다.
수집기가 **지어내기를 거부**한 것입니다.

```
20260901 의 다음 거래일을 몰라 known_at 을 정할 수 없다.
  달력이 20260901 까지밖에 없다.
  왜 세우나: 다음 거래일을 지어내면 아직 열리지 않은 장에 자료를 붙이게 된다.
```

`known_at` 이 `basDt` 의 **다음 거래일**이라, 달력의 마지막 날은 **언제나** 받을 수 없습니다.

In [10]:
달력끝 = 하나("SELECT MAX(bas_dd) FROM trading_calendar")[0]
신원끝 = 하나("SELECT MAX(bas_dd) FROM stock_identity")[0]
시세끝 = 하나("SELECT MAX(bas_dd) FROM daily_price")[0]
print(f"거래일 달력 마지막 : {달력끝}")
print(f"시세 마지막        : {시세끝}")
print(f"신원 마지막        : {신원끝}   ← 달력보다 하루 앞선다 (그래야 known_at 이 선다)")
print()
for r in 표("SELECT bas_dd, rows, fetched_at FROM fetch_log "
            "WHERE bas_dd >= '20260901' ORDER BY bas_dd"):
    print(f"  fetch_log {r[0]}  {r[1]:,}행  {r[2]}")
print()
print("🔴 20260902 를 0행으로 받은 것은 그날 장이 끝나기 전이었기 때문이다.")
print("   다음 시세 수집이 달력을 넓히면 20260901 신원은 저절로 들어온다.")

거래일 달력 마지막 : 20260901
시세 마지막        : 20260901
신원 마지막        : 20260831   ← 달력보다 하루 앞선다 (그래야 known_at 이 선다)

  fetch_log 20260901  2,765행  2026-09-02T09:42:39
  fetch_log 20260902  0행  2026-09-02T14:05:31

🔴 20260902 를 0행으로 받은 것은 그날 장이 끝나기 전이었기 때문이다.
   다음 시세 수집이 달력을 넓히면 20260901 신원은 저절로 들어온다.


---

## 6. `known_at` 은 계산값과 관측값을 섞지 않는다

두 표의 `known_at` 규칙이 **다릅니다.** 이걸 섞으면 어느 행이 어떤 규칙으로 계산됐는지
갈라낼 수 없어, 규칙을 바꿀 때 전량 재수집 말고는 방법이 없어집니다.

In [11]:
for 표이름 in ("stock_identity", "corp_profile"):
    분포 = dict(표(f"SELECT known_rule, COUNT(*) FROM {표이름} GROUP BY known_rule"))
    성격 = "계산값 (포털이 발표 시각을 안 준다)" if 표이름 == "stock_identity" \
        else "관측값 (출처가 유효 시작일을 직접 준다)"
    print(f"{표이름:16s} {분포}")
    print(f"{'':16s} → {성격}")
print()
이른 = 하나("SELECT COUNT(*) FROM stock_identity WHERE known_at <= bas_dd")[0]
print(f"known_at 이 기준일보다 이르거나 같은 행 : {이른}  (0이어야 한다)")
print("  그 날 못 본 자료를 봤다는 뜻이라, 0이 아니면 미래참조다.")

stock_identity   {'basDt+1session': 4197242}
                 → 계산값 (포털이 발표 시각을 안 준다)
corp_profile     {'fstOpegDt': 40569}
                 → 관측값 (출처가 유효 시작일을 직접 준다)



known_at 이 기준일보다 이르거나 같은 행 : 0  (0이어야 한다)
  그 날 못 본 자료를 봤다는 뜻이라, 0이 아니면 미래참조다.


---

## 7. 이 표를 쓸 때 반드시 지킬 것 — 유니버스는 교집합으로

앞 노트북에서 이미 다뤘지만, **가장 조용한 함정**이라 다시 셉니다.

In [12]:
표본날 = "20200102"
포털그날 = {r[0]: r[1] for r in 표(
    "SELECT code, item_nm FROM stock_identity WHERE bas_dd = ?", 표본날)}
늦은것 = []
for 코드, 이름 in 포털그날.items():
    첫날 = 하나("SELECT MIN(bas_dd) FROM daily_price WHERE code = ?", 코드)[0]
    if 첫날 and 첫날 > 표본날:
        늦은것.append((코드, 이름, 첫날))

print(f"포털이 {표본날} 목록이라며 준 종목 : {len(포털그날):,}종")
print(f"  그중 그날 이후에야 상장된 것    : {len(늦은것)}종")
for 코드, 이름, 첫날 in sorted(늦은것, key=lambda x: x[2])[-3:]:
    print(f"    {코드}  {이름:16s} 첫 시세 {첫날}")
print()
print("✅ 이렇게 씁니다")
print("   universe = set(신원_그날) & set(그날_실제_거래된_종목)")
print("🔴 이렇게 쓰면 미래참조입니다 (에러 없이 성능만 좋아집니다)")
print("   universe = 신원_그날")

포털이 20200102 목록이라며 준 종목 : 2,334종
  그중 그날 이후에야 상장된 것    : 33종
    107640  한중엔시에스           첫 시세 20240624
    044990  에이치엔에스하이텍        첫 시세 20241025
    176750  듀켐바이오            첫 시세 20241220

✅ 이렇게 씁니다
   universe = set(신원_그날) & set(그날_실제_거래된_종목)
🔴 이렇게 쓰면 미래참조입니다 (에러 없이 성능만 좋아집니다)
   universe = 신원_그날


---

## 이 노트북의 결론

| 무엇 | 배운 것 |
|---|---|
| **표본은 통과를 증명하지 못한다** | 같은 검증기가 표본에서 ✅, 전량에서 세 곳을 잡았다. 표본이 하필 그 목록을 봤을 뿐이었다 |
| **자리표시자는 모양이 멀쩡하다** | `00010101` 은 여덟 자리 날짜고 `0000000000000` 은 13자리 숫자다. 자릿수 검사로는 안 걸린다 |
| **틀린 짝이 0행보다 비싸다** | 조인이 비면 눈에 띈다. 20개 회사가 하나로 붙으면 그럴듯해서 안 보인다 |
| **거르는 선은 잃는 것과 함께 정한다** | 1900 이면 동화약품이 죽고, 0001 이면 쓰레기가 산다. 1800 이 그 사이다 |
| **기대값은 다른 경로에서** | "우리가 넣은 값 == 우리가 넣은 값" 은 검사가 아니다. KRX 개장일은 우리 자료 밖 사실이다 |
| **못 받는 것과 못 하는 것은 다르다** | 마지막 하루는 결함이 아니라 `known_at` 규칙이 만드는 경계다. 시세가 하루 늘면 풀린다 |

## 이어지는 것

- [`docs/데이터파트/version3.2/공공데이터포털_수집_설계.md`](../../docs/데이터파트/version3.2/공공데이터포털_수집_설계.md) §7 — 검증 전문
- [`docs/erd/version2.2/ERD.md`](../../docs/erd/version2.2/ERD.md) §2.6 · §2.7 — 표 두 개의 칸과 함정
- [#93](https://github.com/devlee328288/Alpha_Stack/issues/93) — 숫자 쪽에 남은 같은 부류 (`0` 이 "0원" 인지 "미기재" 인지)
- [#92](https://github.com/devlee328288/Alpha_Stack/issues/92) — 업종 매핑은 이 다리로도 안 된다 (실측 근거)

In [13]:
conn.close()
print("닫았습니다.")

닫았습니다.
